In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/finqa-rag-lora-ablation

Mounted at /content/drive
/content/drive/MyDrive/finqa-rag-lora-ablation


In [2]:
!pip install datasets transformers -q

In [7]:
import os
import json
import urllib.request

DATA_URLS = {
    'train': 'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/train.json',
    'dev':   'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/dev.json',
    'test':  'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json',
}

RAW_DIR = 'data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

for split, url in DATA_URLS.items():
    out_path = f'{RAW_DIR}/{split}.json'
    if os.path.exists(out_path):
        print(f" {split}.json exist")
        continue
    print(f" download {split}.json ...")
    urllib.request.urlretrieve(url, out_path)
    print(f"save {out_path}")

finqa = {}
for split in ['train', 'dev', 'test']:
    with open(f'{RAW_DIR}/{split}.json', 'r') as f:
        finqa[split] = json.load(f)
    print(f"{split}: {len(finqa[split])} ")

 train.json exist
 dev.json exist
 test.json exist
train: 6251 
dev: 883 
test: 1147 


In [8]:
sample = finqa['train'][0]

print("top:", list(sample.keys()))
print()
print("=" * 60)

import json
text = json.dumps(sample, indent=2, ensure_ascii=False)
print(text[:3000])
print("..." if len(text) > 3000 else "")

top: ['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'qa', 'id', 'table_retrieved', 'text_retrieved', 'table_retrieved_all', 'text_retrieved_all']

{
  "pre_text": [
    "interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .",
    "if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .",
    "foreign currency exposure as more fully described in note 2i .",
    "in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .",
    "dollar-based exposures by entering into forward foreign currency exchange contracts .",
    "the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .",
    "currently , our largest foreign currency exposure is the euro , primarily because our european op

In [9]:
for i in range(5):
    s = finqa['train'][i]
    qa = s.get('qa', {})
    print(f"--- Sample {i} ---")
    print(f"Question: {qa.get('question', 'N/A')}")
    print(f"Answer:   {qa.get('answer', 'N/A')}")
    print(f"Program:  {qa.get('program', 'N/A')}")
    print()

--- Sample 0 ---
Question: what is the the interest expense in 2009?
Answer:   380
Program:  divide(100, 100), divide(3.8, #0)

--- Sample 1 ---
Question: during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
Answer:   
Program:  multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)

--- Sample 2 ---
Question: what was the total operating expenses in 2018 in millions
Answer:   41932
Program:  divide(9896, 23.6%)

--- Sample 3 ---
Question: what percentage of total cash and investments as of dec . 29 2012 was comprised of available-for-sale investments?
Answer:   53%
Program:  divide(14001, 26302)

--- Sample 4 ---
Question: what is the growth rate in net revenue in 2008?
Answer:   -3.2%
Program:  subtract(959.2, 991.1), divide(#0, 991.1)



In [11]:
import numpy as np
def get_context_length(sample):
    pre = ' '.join(sample.get('pre_text', []))
    post = ' '.join(sample.get('post_text', []))
    return len((pre + ' ' + post).split())

lengths = [get_context_length(s) for s in finqa['train']]

print(f"Context Word Count Statistics:")
print(f"  Mean: {np.mean(lengths):.0f} words")
print(f"  Median: {np.median(lengths):.0f} words")
print(f"  Min/Max: {np.min(lengths)} / {np.max(lengths)} words")

Context Word Count Statistics:
  Mean: 632 words
  Median: 622 words
  Min/Max: 12 / 2618 words


In [12]:
print("First 20 answer samples:")
for i in range(20):
    ans = finqa['train'][i]['qa'].get('answer', '')
    print(f"  [{i}] {ans!r}")

First 20 answer samples:
  [0] '380'
  [1] ''
  [2] '41932'
  [3] '53%'
  [4] '-3.2%'
  [5] '56.25%'
  [6] '7.4'
  [7] '63.6%'
  [8] '96.55%'
  [9] '56.6'
  [10] '6.9'
  [11] ''
  [12] '65.3%'
  [13] '0.3%'
  [14] '28%'
  [15] '65.6%'
  [16] '20.2%'
  [17] '12.03%'
  [18] '3.61%'
  [19] '93.4%'


In [13]:
import re

def classify_answer(answer: str, program: str) -> str:
    """Classify samples into percentage / numeric / boolean / other based on answer and program"""
    answer = (answer or '').strip()
    program = (program or '').strip()

    # boolean: program ends with a comparison operation
    boolean_ops = ['greater', 'less', 'equal', 'smaller', 'greater_or_equal']
    last_op = program.split(',')[-1].strip().split('(')[0] if program else ''
    if last_op in boolean_ops:
        return 'boolean'

    # percentage: answer contains %
    if '%' in answer:
        return 'percentage'

    # numeric: answer is a number (may include negative signs, decimals, or commas)
    if re.match(r'^-?[\d,]+\.?\d*$', answer):
        return 'numeric'

    # other: includes empty strings or text-based answers
    return 'other'


def classify_complexity(program: str) -> str:
    """Classify as simple / complex based on the number of steps in the program"""
    if not program:
        return 'simple'
    n_ops = len([op for op in program.split(',') if op.strip()])
    return 'simple' if n_ops <= 2 else 'complex'


# Test the classification functions
test_cases = [
    ('380', 'divide(100, 100), divide(3.8, #0)'),
    ('53%', 'divide(14001, 26302)'),
    ('', 'multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)'),
]
for ans, prog in test_cases:
    print(f"answer={ans!r:10s} -> {classify_answer(ans, prog):10s} | "
          f"complexity={classify_complexity(prog)}")

answer='380'      -> numeric    | complexity=complex
answer='53%'      -> percentage | complexity=simple
answer=''         -> other      | complexity=complex


In [14]:
from collections import Counter

# Calculate distributions on the train set
ans_types = []
complexities = []
combined = []

for s in finqa['train']:
    qa = s.get('qa', {})
    ans = qa.get('answer', '')
    prog = qa.get('program', '')

    at = classify_answer(ans, prog)
    cx = classify_complexity(prog)

    ans_types.append(at)
    complexities.append(cx)
    combined.append(f"{at}_{cx}")

print("=== Answer Type Distribution ===")
for k, v in Counter(ans_types).most_common():
    print(f"  {k:12s}: {v:5d} ({v/len(ans_types)*100:.1f}%)")

print("\n=== Complexity Distribution ===")
for k, v in Counter(complexities).most_common():
    print(f"  {k:12s}: {v:5d} ({v/len(complexities)*100:.1f}%)")

print("\n=== Joint Distribution (6 strata) ===")
for k, v in sorted(Counter(combined).items()):
    print(f"  {k:25s}: {v:5d} ({v/len(combined)*100:.1f}%)")

=== Answer Type Distribution ===
  percentage  :  3508 (56.1%)
  numeric     :  2440 (39.0%)
  other       :   303 (4.8%)

=== Complexity Distribution ===
  simple      :  3717 (59.5%)
  complex     :  2534 (40.5%)

=== Joint Distribution (6 strata) ===
  numeric_complex          :   675 (10.8%)
  numeric_simple           :  1765 (28.2%)
  other_complex            :    82 (1.3%)
  other_simple             :   221 (3.5%)
  percentage_complex       :  1777 (28.4%)
  percentage_simple        :  1731 (27.7%)


In [15]:
# Inspect samples in the 'other' category to decide whether to include them
print("=== 'other' class samples (top 10) ===")
count = 0
for s in finqa['train']:
    qa = s.get('qa', {})
    ans = qa.get('answer', '')
    prog = qa.get('program', '')
    if classify_answer(ans, prog) == 'other':
        print(f"  answer={ans!r}, program={prog!r}")
        count += 1
        if count >= 10:
            break

=== 'other' class samples (top 10) ===
  answer='', program='multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)'
  answer='', program='add(15553, 7376)'
  answer='$ 13 million', program='subtract(78.0, 75.3), subtract(58.0, 49.9), subtract(54.0, 51.8), add(#0, #1), add(#2, #3)'
  answer='yes', program='greater(189.57, 137.82)'
  answer='$ 7.6 million', program='subtract(34.6, 24.8)'
  answer='yes', program='greater(5941210, 4852978)'
  answer='$ 4236 million', program='multiply(10086, 42%)'
  answer='', program='subtract(51%, 20%), divide(343, #0)'
  answer='yes', program='greater(45, 25)'
  answer='no', program='greater(1, 10)'


In [21]:
%%writefile data/preprocess.py
"""
FinQA Data Preprocessing Pipeline

Downloads the original FinQA dataset from GitHub, classifies samples by
answer type (percentage/numeric/boolean) and complexity (simple/complex),
performs stratified sampling with intentional oversampling of boolean
questions for reliable error analysis.

Usage:
    python data/preprocess.py
    python data/preprocess.py --n_samples 500 --seed 42

Output:
    data/raw/{train,dev,test}.json     - original FinQA splits
    data/processed/finqa_500.json       - 500 stratified samples
    data/processed/sampling_stats.json  - sampling statistics for the report
    data/samples/finqa_10_demo.json     - 10-sample demo (committed to git)
"""

import argparse
import json
import os
import random
import re
import urllib.request
from collections import Counter, defaultdict
from typing import Dict, List

# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

DATA_URLS = {
    'train': 'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/train.json',
    'dev':   'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/dev.json',
    'test':  'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json',
}

# Target sample count per stratum (must sum to total N)
STRATUM_QUOTAS = {
    'percentage_simple':  130,
    'percentage_complex': 130,
    'numeric_simple':     130,
    'numeric_complex':     60,
    'boolean_simple':      40,
    'boolean_complex':     10,
}
TOTAL_SAMPLES = sum(STRATUM_QUOTAS.values())  # 500

BOOLEAN_OPS = {'greater', 'less', 'equal', 'smaller',
               'greater_or_equal', 'less_or_equal'}


# -----------------------------------------------------------------------------
# Classification functions
# -----------------------------------------------------------------------------

def classify_answer(answer: str, program: str) -> str:
    """Classify a sample by answer type: percentage / numeric / boolean / other."""
    answer = (answer or '').strip().lower()
    program = (program or '').strip()

    # 1. boolean: explicit yes/no, OR program ends with comparison op
    if answer in ('yes', 'no', 'true', 'false'):
        return 'boolean'
    if program:
        last_op = program.split(',')[-1].strip().split('(')[0].strip()
        if last_op in BOOLEAN_OPS:
            return 'boolean'

    # 2. percentage
    if '%' in answer:
        return 'percentage'

    # 3. numeric (allows $, comma, million/billion suffixes)
    cleaned = answer.replace('$', '').replace(',', '').replace(' ', '')
    cleaned = re.sub(r'(million|billion|thousand|m|bn|k)$', '', cleaned)
    try:
        float(cleaned)
        return 'numeric'
    except (ValueError, TypeError):
        return 'other'


def classify_complexity(program: str) -> str:
    """Simple if program has <=2 ops, complex otherwise."""
    if not program:
        return 'simple'
    n_ops = len([op for op in program.split(',') if op.strip()])
    return 'simple' if n_ops <= 2 else 'complex'


def get_stratum(sample: dict) -> str:
    """Return the stratum label for a sample, e.g. 'percentage_simple'."""
    qa = sample.get('qa', {}) or {}
    ans_type = classify_answer(qa.get('answer', ''), qa.get('program', ''))
    complexity = classify_complexity(qa.get('program', ''))
    return f"{ans_type}_{complexity}"


# -----------------------------------------------------------------------------
# Data loading
# -----------------------------------------------------------------------------

def download_finqa(raw_dir: str = 'data/raw') -> Dict[str, list]:
    """Download FinQA splits from the official GitHub repo if not already present."""
    os.makedirs(raw_dir, exist_ok=True)
    data = {}
    for split, url in DATA_URLS.items():
        path = os.path.join(raw_dir, f'{split}.json')
        if not os.path.exists(path):
            print(f"Downloading {split}.json ...")
            urllib.request.urlretrieve(url, path)
        with open(path, 'r') as f:
            data[split] = json.load(f)
        print(f"  {split}: {len(data[split])} samples")
    return data


# -----------------------------------------------------------------------------
# Stratified sampling
# -----------------------------------------------------------------------------

def stratified_sample(samples: List[dict], quotas: Dict[str, int],
                      seed: int = 42) -> Dict[str, list]:
    """
    Bucket samples by stratum, then randomly draw `quotas[stratum]` from each.
    Returns dict mapping stratum -> list of sampled records.
    Raises if any stratum has fewer than the requested quota.
    """
    rng = random.Random(seed)

    buckets = defaultdict(list)
    for s in samples:
        stratum = get_stratum(s)
        if stratum in quotas:
            buckets[stratum].append(s)

    sampled = {}
    for stratum, quota in quotas.items():
        pool = buckets.get(stratum, [])
        if len(pool) < quota:
            raise ValueError(
                f"Stratum {stratum!r} has only {len(pool)} samples but "
                f"{quota} were requested. Adjust STRATUM_QUOTAS."
            )
        sampled[stratum] = rng.sample(pool, quota)
        print(f"  {stratum:25s}: drew {quota:3d} from pool of {len(pool):4d}")

    return sampled


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', type=int, default=42,
                        help='Random seed for reproducibility')
    parser.add_argument('--raw_dir', default='data/raw')
    parser.add_argument('--out_dir', default='data/processed')
    parser.add_argument('--demo_dir', default='data/samples')
    args = parser.parse_args()

    print("=" * 60)
    print("Step 1: Download FinQA from GitHub")
    print("=" * 60)
    finqa = download_finqa(args.raw_dir)

    print("\n" + "=" * 60)
    print("Step 2: Compute full distribution on train split")
    print("=" * 60)
    full_strata = Counter(get_stratum(s) for s in finqa['train'])
    total = sum(full_strata.values())
    natural_weights = {}
    for stratum in STRATUM_QUOTAS:
        n = full_strata.get(stratum, 0)
        natural_weights[stratum] = n / total if total > 0 else 0
        print(f"  {stratum:25s}: {n:5d} ({natural_weights[stratum]*100:.1f}%)")

    print("\n" + "=" * 60)
    print(f"Step 3: Stratified sample {TOTAL_SAMPLES} from train")
    print("=" * 60)
    sampled_by_stratum = stratified_sample(finqa['train'], STRATUM_QUOTAS,
                                            seed=args.seed)

    # Flatten to single list, attach stratum metadata for downstream use
    flat_samples = []
    for stratum, items in sampled_by_stratum.items():
        for item in items:
            item['_stratum'] = stratum
            flat_samples.append(item)

    print(f"\n  Total sampled: {len(flat_samples)}")

    print("\n" + "=" * 60)
    print("Step 4: Save outputs")
    print("=" * 60)
    os.makedirs(args.out_dir, exist_ok=True)
    os.makedirs(args.demo_dir, exist_ok=True)

    # Main processed file (NOT committed, in .gitignore)
    out_path = os.path.join(args.out_dir, f'finqa_{TOTAL_SAMPLES}.json')
    with open(out_path, 'w') as f:
        json.dump(flat_samples, f, indent=2)
    print(f"  Saved: {out_path}")

    # Sampling statistics (committed, useful for the report)
    stats = {
        'total_samples': TOTAL_SAMPLES,
        'random_seed': args.seed,
        'stratum_quotas': STRATUM_QUOTAS,
        'natural_weights': natural_weights,
        'note': 'Boolean strata are intentionally oversampled '
                '(~10% sampled vs ~3% natural) for reliable error analysis. '
                'Use natural_weights to recompute population-weighted metrics.'
    }
    stats_path = os.path.join(args.out_dir, 'sampling_stats.json')
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"  Saved: {stats_path}")

    # Demo file with 10 samples (committed for TA reproducibility)
    demo = flat_samples[:10]
    demo_path = os.path.join(args.demo_dir, 'finqa_10_demo.json')
    with open(demo_path, 'w') as f:
        json.dump(demo, f, indent=2)
    print(f"  Saved: {demo_path}")

    print("\n✅ Done.")


if __name__ == '__main__':
    main()

Overwriting data/preprocess.py


In [22]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation
!python data/preprocess.py

/content/drive/MyDrive/finqa-rag-lora-ablation
Step 1: Download FinQA from GitHub
  train: 6251 samples
  dev: 883 samples
  test: 1147 samples

Step 2: Compute full distribution on train split
  percentage_simple        :  1731 (27.7%)
  percentage_complex       :  1777 (28.4%)
  numeric_simple           :  1830 (29.3%)
  numeric_complex          :   700 (11.2%)
  boolean_simple           :   107 (1.7%)
  boolean_complex          :    13 (0.2%)

Step 3: Stratified sample 500 from train
  percentage_simple        : drew 130 from pool of 1731
  percentage_complex       : drew 130 from pool of 1777
  numeric_simple           : drew 130 from pool of 1830
  numeric_complex          : drew  60 from pool of  700
  boolean_simple           : drew  40 from pool of  107
  boolean_complex          : drew  10 from pool of   13

  Total sampled: 500

Step 4: Save outputs
  Saved: data/processed/finqa_500.json
  Saved: data/processed/sampling_stats.json
  Saved: data/samples/finqa_10_demo.json

✅ D

In [23]:
import json
from collections import Counter

# Inspect the main output file
with open('data/processed/finqa_500.json') as f:
    data = json.load(f)
print(f"Main file: {len(data)} samples")
print(f"First sample stratum: {data[0]['_stratum']}")
print(f"First sample fields: {list(data[0].keys())[:5]}...")  # Only view the first 5

# Check the sampling statistics
with open('data/processed/sampling_stats.json') as f:
    stats = json.load(f)
print("\nSampling statistics:")
print(json.dumps(stats, indent=2))

print("\nActual count per stratum:")
strata_counts = Counter(s['_stratum'] for s in data)
for k, v in sorted(strata_counts.items()):
    print(f"  {k:25s}: {v}")

Main file: 500 samples
First sample stratum: percentage_simple
First sample fields: ['pre_text', 'post_text', 'filename', 'table_ori', 'table']...

Sampling statistics:
{
  "total_samples": 500,
  "random_seed": 42,
  "stratum_quotas": {
    "percentage_simple": 130,
    "percentage_complex": 130,
    "numeric_simple": 130,
    "numeric_complex": 60,
    "boolean_simple": 40,
    "boolean_complex": 10
  },
  "natural_weights": {
    "percentage_simple": 0.27691569348904177,
    "percentage_complex": 0.2842745160774276,
    "numeric_simple": 0.2927531594944809,
    "numeric_complex": 0.11198208286674133,
    "boolean_simple": 0.017117261238201887,
    "boolean_complex": 0.002079667253239482
  },
  "note": "Boolean strata are intentionally oversampled (~10% sampled vs ~3% natural) for reliable error analysis. Use natural_weights to recompute population-weighted metrics."
}

Actual count per stratum:
  boolean_complex          : 10
  boolean_simple           : 40
  numeric_complex        

In [24]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

gitignore_additions = """
# === FinQA project specific ===
# Raw data downloaded from GitHub (regenerate with: python data/preprocess.py)
data/raw/
data/processed/

# Model checkpoints (saved to Google Drive instead)
checkpoints/
*.bin
*.safetensors

# Wandb logs
wandb/

# Jupyter cache
.ipynb_checkpoints/
"""

with open('.gitignore', 'a') as f:
    f.write(gitignore_additions)

print("Updated .gitignore. Last 20 lines:")
!tail -20 .gitignore

/content/drive/MyDrive/finqa-rag-lora-ablation
Updated .gitignore. Last 20 lines:
__marimo__/

# Streamlit
.streamlit/secrets.toml

# === FinQA project specific ===
# Raw data downloaded from GitHub (regenerate with: python data/preprocess.py)
data/raw/
data/processed/

# Model checkpoints (saved to Google Drive instead)
checkpoints/
*.bin
*.safetensors

# Wandb logs
wandb/

# Jupyter cache
.ipynb_checkpoints/


In [25]:
!git status

Refresh index: 100% (15/15), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	01_data_exploration.ipynb
	data/preprocess.py
	data/samples/finqa_10_demo.json

no changes added to commit (use "git add" and/or "git commit -a")
